
# 2026 国赛 A 题问题 2：最终竞赛版数值分析与论文绘图 Notebook

## Notebook 使用说明

本 Notebook 基于你们当前 **Q2 模型、代码和题目要求** 组织，目的是形成可直接用于竞赛论文的最终分析版本。它将统一完成：

- 正式结果计算；
- 论文图表生成；
- 数值可信性验证；
- 参数敏感性分析；
- 主算法与有限差分法的交叉验证；
- 论文写作提示与章节说明；
- 结果压缩打包。

## 统一约定

1. 所有正式图表均来自真实计算结果；
2. 图片标题、坐标轴、表头统一使用中文；
3. **只导出 PNG 图片**，不生成 SVG；
4. 所有表格统一保存为 CSV / XLSX；
5. Notebook 各章附有“论文写作提示”。

---



# 第一章 研究背景与任务说明

## 1.1 问题 2 的物理背景

问题 2 研究的是：在前 3 小时实测环境边界作用下，圆柱形药材内部温度场与含水率场随时间和半径的变化规律。

模型状态变量为：

- 温度 \(T(r,t)\)
- 含水率 \(C(r,t)\)

其本质是一个一维轴对称径向的变物性、非线性热—湿耦合传递问题。

## 1.2 本 Notebook 的主要任务

- 绘制环境温度和环境水分浓度的原始散点图与线性插值图；
- 计算正式基准解 \(N=1280\)；
- 绘制温度场、含水率场热力图；
- 绘制径向分布图；
- 绘制空间非均匀性演化图；
- 绘制状态依赖扩散系数图；
- 进行 \(h\)、\(h_m\) 敏感性分析；
- 生成 `results2.xlsx`、表 3、表 4；
- 进行网格收敛性分析；
- 将主算法与有限差分法结果进行误差比较；
- 自动压缩全部结果。

## 论文写作提示

这一章不必在正文中单独展开，但建议在“模型建立与求解概述”中用一段简洁文字说明：

- 几何简化为圆柱轴对称径向问题；
- 采用分布参数模型描述内部非均匀性；
- 使用数值方法得到正式结果。


In [ ]:

# ===== 基础依赖、路径与中文绘图设置 =====
from pathlib import Path
import sys
import json
import pickle
import shutil
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.integrate import solve_ivp

PROJECT_ROOT = Path.cwd().resolve()
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data"

if not (SRC_DIR / "q2_solver.py").exists():
    raise FileNotFoundError("请在包含 notebook、src/、data/ 的目录中启动 Jupyter。")

OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIG_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
CACHE_DIR = OUTPUT_DIR / "cache"

for d in [OUTPUT_DIR, FIG_DIR, TABLE_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(SRC_DIR))

from q2_solver import (
    Q2Parameters, SolverConfig, EnvironmentData,
    solve_q2, write_result2, paper_tables, compare_solutions,
    density, heat_capacity, thermal_conductivity, moisture_diffusivity,
    harmonic_mean, jacobian_sparsity
)

# 中文字体与防乱码设置
plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK SC",
    "Microsoft YaHei",
    "SimHei",
    "WenQuanYi Micro Hei",
    "Arial Unicode MS",
    "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 120

ENV_PATH = DATA_DIR / "附件1.xlsx"
assert ENV_PATH.exists(), f"缺少环境数据文件：{ENV_PATH}"

N_BASE = 1280
T_END = 10800.0
BASE_PARAMS = Q2Parameters()

BASE_CONFIG = SolverConfig(
    method="BDF",
    rtol=1e-9,
    atol=1e-11,
    max_step=5.0,
    output_dt=1.0,
    face_average="harmonic",
    segment_at_data_knots=True,
)

SENS_CONFIG = SolverConfig(
    method="BDF",
    rtol=1e-9,
    atol=1e-11,
    max_step=5.0,
    output_dt=60.0,
    face_average="harmonic",
    segment_at_data_knots=True,
)

FDM_OUTPUT_DT = 10.0

def save_figure(fig, stem, dpi=320):
    path = FIG_DIR / f"{stem}.png"
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    print("已保存图片：", path)
    return path

def save_csv(df, name, index=True):
    path = TABLE_DIR / name
    df.to_csv(path, index=index, encoding="utf-8-sig")
    print("已保存表格：", path)
    return path

def solve_cached(cache_name, *, N, config, params=BASE_PARAMS, t_end=T_END, output_dr=0.001):
    cache_path = CACHE_DIR / f"{cache_name}.pkl"
    if cache_path.exists():
        with open(cache_path, "rb") as f:
            sol = pickle.load(f)
        print("已读取缓存：", cache_path)
        return sol

    sol = solve_q2(
        ENV_PATH,
        N=N,
        config=config,
        params=params,
        t_end=t_end,
        output_dr=output_dr,
    )
    with open(cache_path, "wb") as f:
        pickle.dump(sol, f, protocol=pickle.HIGHEST_PROTOCOL)
    print("已保存缓存：", cache_path)
    return sol

def exact_time_indices(full_t, requested_t):
    idx = []
    for x in np.asarray(requested_t, dtype=float):
        hit = np.where(np.isclose(full_t, x, atol=1e-10, rtol=0))[0]
        if hit.size != 1:
            raise ValueError(f"时间 {x} s 未在输出网格中精确找到。")
        idx.append(int(hit[0]))
    return np.asarray(idx, dtype=int)

def exact_radius_indices(full_r, requested_r):
    idx = []
    for x in np.asarray(requested_r, dtype=float):
        hit = np.where(np.isclose(full_r, x, atol=1e-12, rtol=0))[0]
        if hit.size != 1:
            raise ValueError(f"半径 {x} m 未在输出网格中精确找到。")
        idx.append(int(hit[0]))
    return np.asarray(idx, dtype=int)

def volume_average_state(sol, field="C"):
    values = sol.C_state[-1] if field == "C" else sol.T_state[-1]
    return float(np.sum(values * sol.grid.volumes) / np.sum(sol.grid.volumes))

print("工程目录：", PROJECT_ROOT)
print("输出目录：", OUTPUT_DIR)


# 第二章 环境边界处理

## 2.1 章节说明

题目给出了前 3 小时环境边界数据。正式建模中，环境边界直接来自附件 1 的离散数据点，并采用**分段线性插值**构造：

- 环境温度 \(T_\infty(t)\)
- 环境水分浓度 \(C_\infty(t)\)

## 2.2 论文写作提示

在论文中建议将本部分放入“边界条件处理”小节，强调：

- 采用原始测量数据；
- 采用分段线性插值；
- 不进行高阶平滑拟合，避免虚构细节。


In [ ]:

# ===== 读取环境边界数据 =====
env_df = pd.read_excel(ENV_PATH)
required_cols = ["时间", "温度", "水分浓度"]
missing = [c for c in required_cols if c not in env_df.columns]
if missing:
    raise ValueError(f"附件 1 缺少字段：{missing}")

env_3h = (
    env_df.loc[(env_df["时间"] >= 0) & (env_df["时间"] <= T_END), required_cols]
    .copy()
    .sort_values("时间")
    .reset_index(drop=True)
)

t_dense = np.arange(0.0, T_END + 1.0, 1.0)
T_dense = np.interp(t_dense, env_3h["时间"].to_numpy(float), env_3h["温度"].to_numpy(float))
Cinf_dense = np.interp(t_dense, env_3h["时间"].to_numpy(float), env_3h["水分浓度"].to_numpy(float))

temp_raw = pd.DataFrame({
    "时间（s）": env_3h["时间"].to_numpy(float),
    "时间（h）": env_3h["时间"].to_numpy(float) / 3600.0,
    "环境温度（℃）": env_3h["温度"].to_numpy(float),
})
temp_interp = pd.DataFrame({
    "时间（s）": t_dense,
    "时间（h）": t_dense / 3600.0,
    "环境温度（℃）": T_dense,
})
moist_raw = pd.DataFrame({
    "时间（s）": env_3h["时间"].to_numpy(float),
    "时间（h）": env_3h["时间"].to_numpy(float) / 3600.0,
    "环境水分浓度（kg/kg）": env_3h["水分浓度"].to_numpy(float),
})
moist_interp = pd.DataFrame({
    "时间（s）": t_dense,
    "时间（h）": t_dense / 3600.0,
    "环境水分浓度（kg/kg）": Cinf_dense,
})

save_csv(temp_raw, "环境温度原始数据_前3小时.csv", index=False)
save_csv(temp_interp, "环境温度线性插值_前3小时.csv", index=False)
save_csv(moist_raw, "环境水分浓度原始数据_前3小时.csv", index=False)
save_csv(moist_interp, "环境水分浓度线性插值_前3小时.csv", index=False)

In [ ]:

# ===== 图 1-3：Environmental 相关图 =====
fig, ax = plt.subplots(figsize=(7.2, 4.8))
ax.plot(t_dense / 3600.0, T_dense, linewidth=1.5, label="线性插值")
ax.scatter(env_3h["时间"] / 3600.0, env_3h["温度"], s=14, label="原始测量点", zorder=3)
ax.set_xlabel("时间（h）")
ax.set_ylabel("环境温度（℃）")
ax.set_xlim(0, 3)
ax.grid(True, alpha=0.25)
ax.legend(frameon=False)
fig.tight_layout()
save_figure(fig, "图1_前3小时环境温度变化")
plt.show()

fig, ax = plt.subplots(figsize=(7.2, 4.8))
ax.plot(t_dense / 3600.0, Cinf_dense, linewidth=1.5, label="线性插值")
ax.scatter(env_3h["时间"] / 3600.0, env_3h["水分浓度"], s=14, label="原始测量点", zorder=3)
ax.set_xlabel("时间（h）")
ax.set_ylabel("环境水分浓度（kg/kg）")
ax.set_xlim(0, 3)
ax.grid(True, alpha=0.25)
ax.legend(frameon=False)
fig.tight_layout()
save_figure(fig, "图2_前3小时环境水分浓度变化")
plt.show()

fig, axes = plt.subplots(2, 1, figsize=(7.3, 7.2), sharex=True)
axes[0].plot(t_dense / 3600.0, T_dense, linewidth=1.5)
axes[0].scatter(env_3h["时间"] / 3600.0, env_3h["温度"], s=12, zorder=3)
axes[0].set_ylabel("环境温度（℃）")
axes[0].grid(True, alpha=0.25)
axes[1].plot(t_dense / 3600.0, Cinf_dense, linewidth=1.5)
axes[1].scatter(env_3h["时间"] / 3600.0, env_3h["水分浓度"], s=12, zorder=3)
axes[1].set_xlabel("时间（h）")
axes[1].set_ylabel("环境水分浓度（kg/kg）")
axes[1].grid(True, alpha=0.25)
fig.tight_layout()
save_figure(fig, "图3_环境边界变化_Environmental")
plt.show()


# 第三章 正式基准解计算

## 3.1 章节说明

本章计算问题 2 的正式基准解。统一采用：

- 空间离散：圆柱坐标下一维径向有限体积法；
- 时间积分：BDF；
- 正式空间网格：\(N=1280\)；
- 正式输出时间间隔：1 s；
- 正式输出半径间隔：0.1 cm。

## 3.2 论文写作提示

本章在论文中建议归入“数值方法与计算设置”部分，重点说明：

- 使用守恒型离散；
- 网格足够细；
- 后续所有正式图表都基于该基准解。


In [ ]:

# ===== 正式基准解 =====
base_sol = solve_cached(
    "Q2_正式基准解_N1280_输出1s",
    N=N_BASE,
    config=BASE_CONFIG,
    params=BASE_PARAMS,
)

print("时间点数：", len(base_sol.t))
print("半径点数：", len(base_sol.r))
print("3h 时中心/表面温度：", base_sol.T[-1, 0], base_sol.T[-1, -1])
print("3h 时中心/表面含水率：", base_sol.C[-1, 0], base_sol.C[-1, -1])

D_field = moisture_diffusivity(base_sol.C, base_sol.T)


# 第四章 论文主图绘制

## 4.1 章节说明

本章集中生成最适合写入正文的图片，包括：

- Environmental（环境边界图）；
- Radial profiles（径向分布图）；
- Spatial heterogeneity evolution（空间非均匀性演化图）；
- State-dependent diffusivity（状态依赖扩散系数图）；
- 温度和含水率时空热力图。

## 4.2 论文写作提示

建议论文正文优先使用：

1. 环境边界组合图；
2. 温度场/含水率场热力图；
3. 温度/含水率径向分布图；
4. 空间非均匀性演化图；
5. 状态依赖扩散系数图。

这些图可以共同支撑“热场先均匀、水分场后均匀”的建模结论。


In [ ]:

# ===== 图 4-5：温度与含水率热力图 =====
fig, ax = plt.subplots(figsize=(7.4, 5.0))
mesh = ax.pcolormesh(base_sol.t / 3600.0, base_sol.r * 100.0, base_sol.T.T, shading="auto")
cbar = fig.colorbar(mesh, ax=ax)
cbar.set_label("温度（℃）")
ax.set_xlabel("时间（h）")
ax.set_ylabel("半径（cm）")
ax.set_xlim(0, 3)
ax.set_ylim(0, 2)
fig.tight_layout()
save_figure(fig, "图4_温度场时空演化热力图")
plt.show()

fig, ax = plt.subplots(figsize=(7.4, 5.0))
mesh = ax.pcolormesh(base_sol.t / 3600.0, base_sol.r * 100.0, base_sol.C.T, shading="auto")
cbar = fig.colorbar(mesh, ax=ax)
cbar.set_label("含水率（kg/kg）")
ax.set_xlabel("时间（h）")
ax.set_ylabel("半径（cm）")
ax.set_xlim(0, 3)
ax.set_ylim(0, 2)
fig.tight_layout()
save_figure(fig, "图5_含水率场时空演化热力图")
plt.show()

In [ ]:

# ===== 图 6-7：Radial profiles =====
profile_hours = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0])
profile_times = profile_hours * 3600.0
it = exact_time_indices(base_sol.t, profile_times)

T_profiles = pd.DataFrame(
    base_sol.T[it].T,
    index=base_sol.r * 100.0,
    columns=[f"{x:.1f} h" for x in profile_hours],
)
T_profiles.index.name = "半径（cm）"

C_profiles = pd.DataFrame(
    base_sol.C[it].T,
    index=base_sol.r * 100.0,
    columns=[f"{x:.1f} h" for x in profile_hours],
)
C_profiles.index.name = "半径（cm）"

save_csv(T_profiles, "径向温度分布_Radial_profiles.csv", index=True)
save_csv(C_profiles, "径向含水率分布_Radial_profiles.csv", index=True)

fig, ax = plt.subplots(figsize=(7.1, 4.9))
for j, hour in enumerate(profile_hours):
    ax.plot(base_sol.r * 100.0, base_sol.T[it[j]], marker="o", markersize=3, label=f"{hour:.1f} h")
ax.set_xlabel("半径（cm）")
ax.set_ylabel("温度（℃）")
ax.set_xlim(0, 2)
ax.grid(True, alpha=0.25)
ax.legend(frameon=False, ncol=2)
fig.tight_layout()
save_figure(fig, "图6_径向温度分布_Radial_profiles")
plt.show()

fig, ax = plt.subplots(figsize=(7.1, 4.9))
for j, hour in enumerate(profile_hours):
    ax.plot(base_sol.r * 100.0, base_sol.C[it[j]], marker="o", markersize=3, label=f"{hour:.1f} h")
ax.set_xlabel("半径（cm）")
ax.set_ylabel("含水率（kg/kg）")
ax.set_xlim(0, 2)
ax.grid(True, alpha=0.25)
ax.legend(frameon=False, ncol=2)
fig.tight_layout()
save_figure(fig, "图7_径向含水率分布_Radial_profiles")
plt.show()

In [ ]:

# ===== 图 8：Spatial heterogeneity evolution =====
deltaT = base_sol.T[:, -1] - base_sol.T[:, 0]
deltaC = base_sol.C[:, 0] - base_sol.C[:, -1]

peak_T_idx = int(np.argmax(deltaT))
peak_C_idx = int(np.argmax(deltaC))

hetero_df = pd.DataFrame({
    "时间（s）": base_sol.t,
    "时间（h）": base_sol.t / 3600.0,
    "表面-中心温差ΔT（℃）": deltaT,
    "中心-表面含水率差ΔC（kg/kg）": deltaC,
})
save_csv(hetero_df, "空间非均匀性演化_Spatial_heterogeneity_evolution.csv", index=False)

fig, axes = plt.subplots(2, 1, figsize=(7.3, 7.4), sharex=True)
axes[0].plot(base_sol.t / 3600.0, deltaT, linewidth=1.5)
axes[0].scatter(base_sol.t[peak_T_idx] / 3600.0, deltaT[peak_T_idx], s=25, zorder=3)
axes[0].annotate(
    f"峰值：{deltaT[peak_T_idx]:.3f} ℃\n时刻：{base_sol.t[peak_T_idx]/3600.0:.2f} h",
    xy=(base_sol.t[peak_T_idx] / 3600.0, deltaT[peak_T_idx]),
    xytext=(0.58, 0.75),
    textcoords="axes fraction",
    arrowprops=dict(arrowstyle="->"),
)
axes[0].set_ylabel("表面—中心温差 ΔT（℃）")
axes[0].grid(True, alpha=0.25)

axes[1].plot(base_sol.t / 3600.0, deltaC, linewidth=1.5)
axes[1].scatter(base_sol.t[peak_C_idx] / 3600.0, deltaC[peak_C_idx], s=25, zorder=3)
axes[1].annotate(
    f"峰值：{deltaC[peak_C_idx]:.3f} kg/kg\n时刻：{base_sol.t[peak_C_idx]/3600.0:.2f} h",
    xy=(base_sol.t[peak_C_idx] / 3600.0, deltaC[peak_C_idx]),
    xytext=(0.58, 0.70),
    textcoords="axes fraction",
    arrowprops=dict(arrowstyle="->"),
)
axes[1].set_xlabel("时间（h）")
axes[1].set_ylabel("中心—表面含水率差 ΔC（kg/kg）")
axes[1].grid(True, alpha=0.25)

fig.tight_layout()
save_figure(fig, "图8_空间非均匀性演化_Spatial_heterogeneity_evolution")
plt.show()

In [ ]:

# ===== 图 9：State-dependent diffusivity =====
D_plot = D_field * 1e9

D_df = pd.DataFrame(D_plot, index=base_sol.t, columns=base_sol.r * 100.0)
D_df.index.name = "时间（s）"
save_csv(D_df, "状态依赖扩散系数场_State_dependent_diffusivity.csv", index=True)

fig, ax = plt.subplots(figsize=(7.4, 5.0))
mesh = ax.pcolormesh(base_sol.t / 3600.0, base_sol.r * 100.0, D_plot.T, shading="auto")
cbar = fig.colorbar(mesh, ax=ax)
cbar.set_label("有效扩散系数（10^-9 m²/s）")
ax.set_xlabel("时间（h）")
ax.set_ylabel("半径（cm）")
ax.set_xlim(0, 3)
ax.set_ylim(0, 2)
fig.tight_layout()
save_figure(fig, "图9_状态依赖扩散系数_State_dependent_diffusivity")
plt.show()


# 第五章 \(h\)、\(h_m\) 参数敏感性分析

## 5.1 章节说明

本章对

- 表面对流换热系数 \(h\)
- 表面传质系数 \(h_m\)

分别做 \(\pm 20\%\)、每 5% 一档的单因素敏感性分析。

## 5.2 论文写作提示

正文中建议重点回答：

- \(h\) 的变化对温度结果影响有多大？
- \(h_m\) 的变化对含水率结果影响有多大？
- 哪个参数更影响“外干内湿”的非均匀性？


In [ ]:

# ===== 敏感性分析 =====
FACTORS = np.round(np.arange(0.80, 1.2001, 0.05), 2)

def baseline_on_times(times):
    ii = exact_time_indices(base_sol.t, times)
    return base_sol.T[ii], base_sol.C[ii]

def baseline_sensitivity_view():
    ii = exact_time_indices(base_sol.t, np.arange(0.0, T_END + 60.0, 60.0))
    class View:
        pass
    v = View()
    v.t = base_sol.t[ii]
    v.r = base_sol.r
    v.T = base_sol.T[ii]
    v.C = base_sol.C[ii]
    v.T_state = base_sol.T_state
    v.C_state = base_sol.C_state
    v.grid = base_sol.grid
    return v

def sensitivity_metrics(sol, parameter, factor, parameter_value):
    Tb, Cb = baseline_on_times(sol.t)
    dT = sol.T - Tb
    dC = sol.C - Cb

    Tmean = volume_average_state(sol, "T")
    Cmean = volume_average_state(sol, "C")
    Tmean0 = volume_average_state(base_sol, "T")
    Cmean0 = volume_average_state(base_sol, "C")

    return {
        "参数": parameter,
        "倍率": float(factor),
        "相对变化（%）": float((factor - 1.0) * 100.0),
        "参数值": float(parameter_value),
        "网格数N": int(sol.grid.N),
        "3h中心温度（℃）": float(sol.T[-1, 0]),
        "3h表面温度（℃）": float(sol.T[-1, -1]),
        "3h体积平均温度（℃）": Tmean,
        "3h中心含水率（kg/kg）": float(sol.C[-1, 0]),
        "3h表面含水率（kg/kg）": float(sol.C[-1, -1]),
        "3h体积平均含水率（kg/kg）": Cmean,
        "3h表面-中心温差（℃）": float(sol.T[-1, -1] - sol.T[-1, 0]),
        "3h中心-表面含水率差（kg/kg）": float(sol.C[-1, 0] - sol.C[-1, -1]),
        "相对基准最大温度偏差（℃）": float(np.max(np.abs(dT))),
        "相对基准最大含水率偏差（kg/kg）": float(np.max(np.abs(dC))),
        "体积平均温度相对变化（%）": float((Tmean - Tmean0) / Tmean0 * 100.0),
        "体积平均含水率相对变化（%）": float((Cmean - Cmean0) / Cmean0 * 100.0),
    }

def run_one_sensitivity(parameter, factor):
    if np.isclose(factor, 1.0):
        param_value = BASE_PARAMS.h if parameter == "h" else BASE_PARAMS.hm
        return sensitivity_metrics(baseline_sensitivity_view(), parameter, factor, param_value)

    kwargs = dict(BASE_PARAMS.__dict__)
    kwargs[parameter] = kwargs[parameter] * float(factor)
    params = Q2Parameters(**kwargs)
    tag = f"敏感性_{parameter}_{factor:.2f}_N1280"
    sol = solve_cached(tag, N=N_BASE, config=SENS_CONFIG, params=params)
    return sensitivity_metrics(sol, parameter, factor, getattr(params, parameter))

def run_sensitivity_series(parameter):
    progress_path = TABLE_DIR / f"_敏感性进度_{parameter}.csv"
    existing = {}

    if progress_path.exists():
        old = pd.read_csv(progress_path)
        for _, row in old.iterrows():
            existing[round(float(row["倍率"]), 2)] = row.to_dict()

    rows = []
    for factor in FACTORS:
        key = round(float(factor), 2)
        if key in existing:
            row = existing[key]
            print(f"复用已有结果：{parameter}, 倍率={factor:.2f}")
        else:
            print(f"正在计算：{parameter}, 倍率={factor:.2f}, N={N_BASE}")
            row = run_one_sensitivity(parameter, float(factor))
            existing[key] = row
            pd.DataFrame(list(existing.values())).sort_values("倍率").to_csv(
                progress_path, index=False, encoding="utf-8-sig"
            )
        rows.append(row)

    return pd.DataFrame(rows).sort_values("倍率").reset_index(drop=True)

sens_h = run_sensitivity_series("h")
sens_hm = run_sensitivity_series("hm")

save_csv(sens_h, "敏感性分析_h_N1280.csv", index=False)
save_csv(sens_hm, "敏感性分析_hm_N1280.csv", index=False)

with pd.ExcelWriter(TABLE_DIR / "敏感性分析汇总_N1280.xlsx", engine="openpyxl") as writer:
    sens_h.to_excel(writer, sheet_name="h敏感性", index=False)
    sens_hm.to_excel(writer, sheet_name="hm敏感性", index=False)

In [ ]:

# ===== 图 10-13：敏感性分析图 =====
x = sens_h["相对变化（%）"]

fig, ax = plt.subplots(figsize=(7.2, 4.8))
ax.plot(x, sens_h["3h中心温度（℃）"], marker="o", label="中心")
ax.plot(x, sens_h["3h表面温度（℃）"], marker="s", label="表面")
ax.plot(x, sens_h["3h体积平均温度（℃）"], marker="^", label="体积平均")
ax.axvline(0, linewidth=0.9, linestyle="--")
ax.set_xlabel("h 相对基准变化（%）")
ax.set_ylabel("温度（℃）")
ax.grid(True, alpha=0.25)
ax.legend(frameon=False)
fig.tight_layout()
save_figure(fig, "图10_h对温度结果的敏感性")
plt.show()

fig, ax = plt.subplots(figsize=(7.2, 4.8))
ax.plot(x, sens_h["3h中心含水率（kg/kg）"], marker="o", label="中心")
ax.plot(x, sens_h["3h表面含水率（kg/kg）"], marker="s", label="表面")
ax.plot(x, sens_h["3h体积平均含水率（kg/kg）"], marker="^", label="体积平均")
ax.axvline(0, linewidth=0.9, linestyle="--")
ax.set_xlabel("h 相对基准变化（%）")
ax.set_ylabel("含水率（kg/kg）")
ax.grid(True, alpha=0.25)
ax.legend(frameon=False)
fig.tight_layout()
save_figure(fig, "图11_h对含水率结果的敏感性")
plt.show()

x = sens_hm["相对变化（%）"]

fig, ax = plt.subplots(figsize=(7.2, 4.8))
ax.plot(x, sens_hm["3h中心温度（℃）"], marker="o", label="中心")
ax.plot(x, sens_hm["3h表面温度（℃）"], marker="s", label="表面")
ax.plot(x, sens_hm["3h体积平均温度（℃）"], marker="^", label="体积平均")
ax.axvline(0, linewidth=0.9, linestyle="--")
ax.set_xlabel("h_m 相对基准变化（%）")
ax.set_ylabel("温度（℃）")
ax.grid(True, alpha=0.25)
ax.legend(frameon=False)
fig.tight_layout()
save_figure(fig, "图12_hm对温度结果的敏感性")
plt.show()

fig, ax = plt.subplots(figsize=(7.2, 4.8))
ax.plot(x, sens_hm["3h中心含水率（kg/kg）"], marker="o", label="中心")
ax.plot(x, sens_hm["3h表面含水率（kg/kg）"], marker="s", label="表面")
ax.plot(x, sens_hm["3h体积平均含水率（kg/kg）"], marker="^", label="体积平均")
ax.axvline(0, linewidth=0.9, linestyle="--")
ax.set_xlabel("h_m 相对基准变化（%）")
ax.set_ylabel("含水率（kg/kg）")
ax.grid(True, alpha=0.25)
ax.legend(frameon=False)
fig.tight_layout()
save_figure(fig, "图13_hm对含水率结果的敏感性")
plt.show()


# 第六章 `results2.xlsx`、表 3 与表 4 导出

## 6.1 章节说明

本章将正式基准解导出为 `results2.xlsx`，并进一步生成论文中的表 3 和表 4。

## 6.2 论文写作提示

建议在表格后补一小段文字，说明：

- 温度分层较快减弱；
- 含水率分层持续更久；
- 表格结论与热力图、径向分布图相互印证。


In [ ]:

# ===== 导出 results2.xlsx、表3、表4 =====
results2_path = OUTPUT_DIR / "results2.xlsx"
write_result2(base_sol, results2_path, template_path=None, decimals=4)

table3_T, table4_C = paper_tables(base_sol)
table3_T.index.name = "时间（h）"
table4_C.index.name = "时间（h）"

save_csv(table3_T, "表3_温度结果.csv", index=True)
save_csv(table4_C, "表4_含水率结果.csv", index=True)

with pd.ExcelWriter(TABLE_DIR / "表3表4汇总.xlsx", engine="openpyxl") as writer:
    table3_T.to_excel(writer, sheet_name="表3_温度")
    table4_C.to_excel(writer, sheet_name="表4_含水率")

assert len(base_sol.t) == 10801
assert len(base_sol.r) == 21
assert np.allclose(np.diff(base_sol.t), 1.0)
assert np.isclose(base_sol.r[0], 0.0)
assert np.isclose(base_sol.r[-1], 0.02)
assert np.all(np.isfinite(base_sol.T))
assert np.all(np.isfinite(base_sol.C))
assert np.all(base_sol.C > 0)


# 第七章 网格收敛性分析

## 7.1 章节说明

本章通过比较 \(N=160,320,640,1280\) 四组空间网格，说明正式结果对空间网格划分不敏感。

## 7.2 论文写作提示

建议在论文中简要说明：

- 随着网格加密，误差持续下降；
- \(N=1280\) 可以视为数值上充分稳定；
- 正式结果与网格划分无关。


In [ ]:

# ===== 网格收敛性分析 =====
GRID_NS = [160, 320, 640, 1280]
grid_solutions = {1280: base_sol}

for N in GRID_NS[:-1]:
    grid_solutions[N] = solve_cached(
        f"网格收敛_N{N}_输出1s",
        N=N,
        config=BASE_CONFIG,
        params=BASE_PARAMS,
    )

rows = []
for N in GRID_NS:
    sol = grid_solutions[N]
    if N == 1280:
        dT = np.zeros_like(base_sol.T)
        dC = np.zeros_like(base_sol.C)
    else:
        dT = sol.T - base_sol.T
        dC = sol.C - base_sol.C

    rows.append({
        "网格数N": N,
        "径向步长（m）": BASE_PARAMS.R / N,
        "径向步长（cm）": 100.0 * BASE_PARAMS.R / N,
        "相对N=1280最大温度误差（℃）": float(np.max(np.abs(dT))),
        "相对N=1280温度RMSE（℃）": float(np.sqrt(np.mean(dT**2))),
        "相对N=1280最大含水率误差（kg/kg）": float(np.max(np.abs(dC))),
        "相对N=1280含水率RMSE（kg/kg）": float(np.sqrt(np.mean(dC**2))),
        "3h中心温度（℃）": float(sol.T[-1, 0]),
        "3h表面温度（℃）": float(sol.T[-1, -1]),
        "3h中心含水率（kg/kg）": float(sol.C[-1, 0]),
        "3h表面含水率（kg/kg）": float(sol.C[-1, -1]),
    })

grid_conv = pd.DataFrame(rows)
save_csv(grid_conv, "网格收敛性分析_相对N1280.csv", index=False)

pair_rows = []
for Na, Nb in zip(GRID_NS[:-1], GRID_NS[1:]):
    cmp = compare_solutions(grid_solutions[Na], grid_solutions[Nb])
    pair_rows.append({
        "粗网格N": Na,
        "细网格N": Nb,
        "最大温度差（℃）": cmp["max_abs_T"],
        "最大含水率差（kg/kg）": cmp["max_abs_C"],
        "温度差峰值时刻（s）": cmp["T_location"][0],
        "温度差峰值半径（cm）": cmp["T_location"][1] * 100.0,
        "含水率差峰值时刻（s）": cmp["C_location"][0],
        "含水率差峰值半径（cm）": cmp["C_location"][1] * 100.0,
    })

grid_pairwise = pd.DataFrame(pair_rows)
grid_pairwise["温度误差缩减比"] = np.nan
grid_pairwise["含水率误差缩减比"] = np.nan
for i in range(len(grid_pairwise) - 1):
    grid_pairwise.loc[i, "温度误差缩减比"] = (
        grid_pairwise.loc[i, "最大温度差（℃）"] / grid_pairwise.loc[i + 1, "最大温度差（℃）"]
    )
    grid_pairwise.loc[i, "含水率误差缩减比"] = (
        grid_pairwise.loc[i, "最大含水率差（kg/kg）"] / grid_pairwise.loc[i + 1, "最大含水率差（kg/kg）"]
    )

save_csv(grid_pairwise, "网格收敛性分析_相邻网格比较.csv", index=False)

In [ ]:

# ===== 图 14-15：网格收敛图 =====
plot_df = grid_conv.loc[grid_conv["网格数N"] < 1280].copy()

fig, ax = plt.subplots(figsize=(6.8, 4.8))
ax.loglog(plot_df["网格数N"], plot_df["相对N=1280最大温度误差（℃）"], marker="o", label="最大绝对误差")
ax.loglog(plot_df["网格数N"], plot_df["相对N=1280温度RMSE（℃）"], marker="s", label="RMSE")
ax.set_xlabel("径向网格数 N")
ax.set_ylabel("误差（℃）")
ax.grid(True, which="both", alpha=0.25)
ax.legend(frameon=False)
fig.tight_layout()
save_figure(fig, "图14_温度结果的网格收敛性")
plt.show()

fig, ax = plt.subplots(figsize=(6.8, 4.8))
ax.loglog(plot_df["网格数N"], plot_df["相对N=1280最大含水率误差（kg/kg）"], marker="o", label="最大绝对误差")
ax.loglog(plot_df["网格数N"], plot_df["相对N=1280含水率RMSE（kg/kg）"], marker="s", label="RMSE")
ax.set_xlabel("径向网格数 N")
ax.set_ylabel("误差（kg/kg）")
ax.grid(True, which="both", alpha=0.25)
ax.legend(frameon=False)
fig.tight_layout()
save_figure(fig, "图15_含水率结果的网格收敛性")
plt.show()


# 第八章 主算法与有限差分法误差分析

## 8.1 章节说明

为了增强数值结果的可信性，本章独立实现一个 **node-centered 有限差分法（FDM）**，并与主算法 FVM 进行对比。

两者共享相同的物理模型、边界条件、正式网格数 \(N=1280\) 和 BDF 时间积分，但空间离散方式不同。

## 8.2 论文写作提示

在正文或附录中建议强调：

- FVM 与 FDM 的结果高度一致；
- 结果不依赖于单一离散格式；
- 这属于独立交叉验证。


In [ ]:

# ===== 独立 node-centered FDM =====
def fdm_surface_values(T, C, t, r_nodes, env, params, config):
    dr = r_nodes[1] - r_nodes[0]
    Tinf = float(env.T_inf(t))
    Cinf = float(env.C_inf(t))
    Tp, Tpp = float(T[-1]), float(T[-2])
    Cp, Cpp = float(C[-1]), float(C[-2])
    Cs = Cp
    Ts = Tp
    omega = config.surface_relaxation

    for _ in range(1, config.surface_max_iter + 1):
        ks = float(thermal_conductivity(Cs))
        Ts_raw = (ks * (4.0 * Tp - Tpp) + 2.0 * dr * params.h * Tinf) / (3.0 * ks + 2.0 * dr * params.h)
        Ds = float(moisture_diffusivity(Cs, Ts_raw))
        Cs_raw = (Ds * (4.0 * Cp - Cpp) + 2.0 * dr * params.hm * Cinf) / (3.0 * Ds + 2.0 * dr * params.hm)
        Ts_new = omega * Ts_raw + (1.0 - omega) * Ts
        Cs_new = omega * Cs_raw + (1.0 - omega) * Cs

        if Cs_new <= 0 or not (np.isfinite(Ts_new) and np.isfinite(Cs_new)):
            raise FloatingPointError("FDM 表面状态无效。")
        if abs(Ts_new - Ts) <= config.surface_tol_T and abs(Cs_new - Cs) <= config.surface_tol_C:
            return float(Ts_new), float(Cs_new)
        Ts, Cs = Ts_new, Cs_new

    raise RuntimeError(f"FDM 表面 Picard 在 t={t:.6g} s 未收敛。")

def fdm_rhs(t, y, r_nodes, env, params, config):
    M = len(r_nodes)
    dr = r_nodes[1] - r_nodes[0]
    T = np.asarray(y[:M], dtype=float)
    C = np.asarray(y[M:], dtype=float)

    if np.any(C <= 0) or not np.all(np.isfinite(y)):
        raise FloatingPointError("FDM 状态无效。")

    rho = density(C)
    cp = heat_capacity(C)
    k = thermal_conductivity(C)
    D = moisture_diffusivity(C, T)

    Ts, Cs = fdm_surface_values(T, C, t, r_nodes, env, params, config)
    ks = float(thermal_conductivity(Cs))
    Ds = float(moisture_diffusivity(Cs, Ts))

    dT = np.empty(M, dtype=float)
    dC = np.empty(M, dtype=float)

    k01 = float(harmonic_mean(np.array([k[0]]), np.array([k[1]]))[0])
    D01 = float(harmonic_mean(np.array([D[0]]), np.array([D[1]]))[0])
    dT[0] = 4.0 * k01 * (T[1] - T[0]) / dr**2 / (rho[0] * cp[0])
    dC[0] = 4.0 * D01 * (C[1] - C[0]) / dr**2

    if M > 2:
        i = np.arange(1, M - 1)
        ri = r_nodes[i]
        rp = ri + 0.5 * dr
        rm = ri - 0.5 * dr
        kp = harmonic_mean(k[i], k[i + 1])
        km = harmonic_mean(k[i - 1], k[i])
        Dp = harmonic_mean(D[i], D[i + 1])
        Dm = harmonic_mean(D[i - 1], D[i])
        LT = (rp * kp * (T[i + 1] - T[i]) / dr - rm * km * (T[i] - T[i - 1]) / dr) / (ri * dr)
        LC = (rp * Dp * (C[i + 1] - C[i]) / dr - rm * Dm * (C[i] - C[i - 1]) / dr) / (ri * dr)
        dT[i] = LT / (rho[i] * cp[i])
        dC[i] = LC

    i = M - 1
    ri = r_nodes[i]
    rp = ri + 0.5 * dr
    rm = ri - 0.5 * dr
    kp = float(harmonic_mean(np.array([k[i]]), np.array([ks]))[0])
    km = float(harmonic_mean(np.array([k[i - 1]]), np.array([k[i]]))[0])
    Dp = float(harmonic_mean(np.array([D[i]]), np.array([Ds]))[0])
    Dm = float(harmonic_mean(np.array([D[i - 1]]), np.array([D[i]]))[0])
    LT = (rp * kp * (Ts - T[i]) / dr - rm * km * (T[i] - T[i - 1]) / dr) / (ri * dr)
    LC = (rp * Dp * (Cs - C[i]) / dr - rm * Dm * (C[i] - C[i - 1]) / dr) / (ri * dr)
    dT[i] = LT / (rho[i] * cp[i])
    dC[i] = LC

    return np.concatenate([dT, dC])

def fdm_sample(T, C, t, r_nodes, target_r, env, params, config):
    if np.isclose(t, 0.0):
        Ts, Cs = params.T0_C, params.C0
    else:
        Ts, Cs = fdm_surface_values(T, C, t, r_nodes, env, params, config)

    r_aug = np.concatenate([r_nodes, [params.R]])
    T_aug = np.concatenate([T, [Ts]])
    C_aug = np.concatenate([C, [Cs]])
    return np.interp(target_r, r_aug, T_aug), np.interp(target_r, r_aug, C_aug)

def solve_fdm_reference(N=1280, output_dt=10.0, params=BASE_PARAMS, config=BASE_CONFIG, t_end=T_END):
    cache_path = CACHE_DIR / f"FDM参考解_N{N}_输出{output_dt:g}s.npz"
    if cache_path.exists():
        z = np.load(cache_path)
        print("已读取 FDM 缓存：", cache_path)
        return z["t"], z["r"], z["T"], z["C"]

    env = EnvironmentData.from_excel(ENV_PATH)
    dr = params.R / N
    r_nodes = np.arange(N, dtype=float) * dr
    target_r = np.linspace(0.0, params.R, 21)

    y = np.concatenate([np.full(N, params.T0_C, dtype=float), np.full(N, params.C0, dtype=float)])
    t_out = np.arange(0.0, t_end + output_dt, output_dt)
    if not np.isclose(t_out[-1], t_end):
        t_out = np.append(t_out[t_out < t_end], t_end)

    knots = env.t[(env.t > 0.0) & (env.t < t_end)]
    edges = np.concatenate([[0.0], knots, [t_end]])
    sparsity = jacobian_sparsity(N)

    all_t, all_T, all_C = [], [], []
    for s in range(len(edges) - 1):
        a = float(edges[s]); b = float(edges[s + 1])
        tev = t_out[(t_out >= a - 1e-12) & (t_out <= b + 1e-12)]
        if s > 0:
            tev = tev[tev > a + 1e-12]
        if tev.size == 0 or tev[-1] < b - 1e-12:
            tev = np.concatenate([tev, [b]])
        if s == 0 and (tev.size == 0 or tev[0] > a + 1e-12):
            tev = np.concatenate([[a], tev])

        rhs = lambda tt, yy: fdm_rhs(tt, yy, r_nodes, env, params, config)
        ivp = solve_ivp(
            rhs, (a, b), y, method="BDF", t_eval=tev,
            rtol=config.rtol, atol=config.atol, max_step=config.max_step,
            jac_sparsity=sparsity,
        )
        if not ivp.success:
            raise RuntimeError(f"FDM BDF 在区间 [{a}, {b}] s 上失败：{ivp.message}")

        for j, tt in enumerate(ivp.t):
            Tsamp, Csamp = fdm_sample(ivp.y[:N, j], ivp.y[N:, j], float(tt), r_nodes, target_r, env, params, config)
            all_t.append(float(tt))
            all_T.append(Tsamp)
            all_C.append(Csamp)

        y = ivp.y[:, -1].copy()

    all_t = np.asarray(all_t)
    all_T = np.asarray(all_T)
    all_C = np.asarray(all_C)
    order = np.argsort(all_t)
    all_t, all_T, all_C = all_t[order], all_T[order], all_C[order]

    _, unique_idx = np.unique(np.round(all_t, 10), return_index=True)
    unique_idx = np.sort(unique_idx)
    all_t, all_T, all_C = all_t[unique_idx], all_T[unique_idx], all_C[unique_idx]

    keep = np.array([np.any(np.isclose(t_out, tt, atol=1e-10, rtol=0)) for tt in all_t])
    all_t, all_T, all_C = all_t[keep], all_T[keep], all_C[keep]

    np.savez_compressed(cache_path, t=all_t, r=target_r, T=all_T, C=all_C)
    print("已保存 FDM 缓存：", cache_path)
    return all_t, target_r, all_T, all_C

In [ ]:

# ===== FVM 与 FDM 误差统计 =====
fdm_t, fdm_r, fdm_T, fdm_C = solve_fdm_reference(N=N_BASE, output_dt=FDM_OUTPUT_DT)

idx_fvm_t = exact_time_indices(base_sol.t, fdm_t)
idx_fvm_r = exact_radius_indices(base_sol.r, fdm_r)
fvm_T = base_sol.T[np.ix_(idx_fvm_t, idx_fvm_r)]
fvm_C = base_sol.C[np.ix_(idx_fvm_t, idx_fvm_r)]

err_T = fdm_T - fvm_T
err_C = fdm_C - fvm_C
abs_T = np.abs(err_T)
abs_C = np.abs(err_C)

summary = pd.DataFrame([
    {
        "变量": "温度",
        "FVM网格数N": N_BASE,
        "FDM网格数N": N_BASE,
        "FDM输出时间间隔（s）": FDM_OUTPUT_DT,
        "最大绝对误差": float(abs_T.max()),
        "平均绝对误差": float(abs_T.mean()),
        "RMSE": float(np.sqrt(np.mean(err_T**2))),
        "参考解值域": float(fvm_T.max() - fvm_T.min()),
        "归一化RMSE（%）": float(np.sqrt(np.mean(err_T**2)) / (fvm_T.max() - fvm_T.min()) * 100.0),
    },
    {
        "变量": "含水率",
        "FVM网格数N": N_BASE,
        "FDM网格数N": N_BASE,
        "FDM输出时间间隔（s）": FDM_OUTPUT_DT,
        "最大绝对误差": float(abs_C.max()),
        "平均绝对误差": float(abs_C.mean()),
        "RMSE": float(np.sqrt(np.mean(err_C**2))),
        "参考解值域": float(fvm_C.max() - fvm_C.min()),
        "归一化RMSE（%）": float(np.sqrt(np.mean(err_C**2)) / (fvm_C.max() - fvm_C.min()) * 100.0),
    },
])
save_csv(summary, "FVM与FDM误差汇总.csv", index=False)

rep_times = np.array([1800, 3600, 5400, 7200, 9000, 10800], dtype=float)
rep_radii = np.array([0.0, 0.005, 0.010, 0.015, 0.020], dtype=float)

it_fdm = exact_time_indices(fdm_t, rep_times)
ir_fdm = exact_radius_indices(fdm_r, rep_radii)
it_fvm = exact_time_indices(base_sol.t, rep_times)
ir_fvm = exact_radius_indices(base_sol.r, rep_radii)

rows_T, rows_C = [], []
for ih, tt in enumerate(rep_times):
    for jr, rr in enumerate(rep_radii):
        tf = float(base_sol.T[it_fvm[ih], ir_fvm[jr]])
        td = float(fdm_T[it_fdm[ih], ir_fdm[jr]])
        cf = float(base_sol.C[it_fvm[ih], ir_fvm[jr]])
        cd = float(fdm_C[it_fdm[ih], ir_fdm[jr]])
        rows_T.append({
            "时间（h）": tt / 3600.0,
            "半径（cm）": rr * 100.0,
            "FVM温度（℃）": tf,
            "FDM温度（℃）": td,
            "绝对误差（℃）": abs(td - tf),
            "相对误差（%）": abs(td - tf) / max(abs(tf), 1e-14) * 100.0,
        })
        rows_C.append({
            "时间（h）": tt / 3600.0,
            "半径（cm）": rr * 100.0,
            "FVM含水率（kg/kg）": cf,
            "FDM含水率（kg/kg）": cd,
            "绝对误差（kg/kg）": abs(cd - cf),
            "相对误差（%）": abs(cd - cf) / max(abs(cf), 1e-14) * 100.0,
        })

fdm_point_T = pd.DataFrame(rows_T)
fdm_point_C = pd.DataFrame(rows_C)
save_csv(fdm_point_T, "FVM与FDM温度逐点误差表.csv", index=False)
save_csv(fdm_point_C, "FVM与FDM含水率逐点误差表.csv", index=False)

with pd.ExcelWriter(TABLE_DIR / "FVM与FDM误差分析.xlsx", engine="openpyxl") as writer:
    summary.to_excel(writer, sheet_name="汇总", index=False)
    fdm_point_T.to_excel(writer, sheet_name="温度", index=False)
    fdm_point_C.to_excel(writer, sheet_name="含水率", index=False)

In [ ]:

# ===== 图 16-19：FVM 与 FDM 误差图 =====
fig, ax = plt.subplots(figsize=(7.4, 5.0))
mesh = ax.pcolormesh(fdm_t / 3600.0, fdm_r * 100.0, abs_T.T, shading="auto")
cbar = fig.colorbar(mesh, ax=ax)
cbar.set_label("温度绝对误差（℃）")
ax.set_xlabel("时间（h）")
ax.set_ylabel("半径（cm）")
ax.set_xlim(0, 3)
ax.set_ylim(0, 2)
fig.tight_layout()
save_figure(fig, "图16_FVM与FDM温度误差热力图")
plt.show()

fig, ax = plt.subplots(figsize=(7.4, 5.0))
mesh = ax.pcolormesh(fdm_t / 3600.0, fdm_r * 100.0, abs_C.T, shading="auto")
cbar = fig.colorbar(mesh, ax=ax)
cbar.set_label("含水率绝对误差（kg/kg）")
ax.set_xlabel("时间（h）")
ax.set_ylabel("半径（cm）")
ax.set_xlim(0, 3)
ax.set_ylim(0, 2)
fig.tight_layout()
save_figure(fig, "图17_FVM与FDM含水率误差热力图")
plt.show()

max_err_T_time = abs_T.max(axis=1)
max_err_C_time = abs_C.max(axis=1)

error_envelope = pd.DataFrame({
    "时间（s）": fdm_t,
    "时间（h）": fdm_t / 3600.0,
    "跨半径最大温度绝对误差（℃）": max_err_T_time,
    "跨半径最大含水率绝对误差（kg/kg）": max_err_C_time,
})
save_csv(error_envelope, "FVM与FDM误差包络随时间变化.csv", index=False)

fig, ax = plt.subplots(figsize=(7.1, 4.8))
ax.plot(fdm_t / 3600.0, max_err_T_time)
ax.set_xlabel("时间（h）")
ax.set_ylabel("跨半径最大温度误差（℃）")
ax.set_xlim(0, 3)
ax.grid(True, alpha=0.25)
fig.tight_layout()
save_figure(fig, "图18_FVM与FDM温度误差包络")
plt.show()

fig, ax = plt.subplots(figsize=(7.1, 4.8))
ax.plot(fdm_t / 3600.0, max_err_C_time)
ax.set_xlabel("时间（h）")
ax.set_ylabel("跨半径最大含水率误差（kg/kg）")
ax.set_xlim(0, 3)
ax.grid(True, alpha=0.25)
fig.tight_layout()
save_figure(fig, "图19_FVM与FDM含水率误差包络")
plt.show()


# 第九章 输出检查、压缩与打包

## 9.1 章节说明

本章自动检查输出结构，并给出压缩命令，最后生成统一压缩包。

## 9.2 论文写作提示

这一章通常不写入论文正文，但对于最终归档非常重要。建议保留：

- `results2.xlsx`
- 正文 PNG 图片
- 所有支撑表格
- 一个统一结果压缩包


In [ ]:

# ===== 输出检查 =====
checks = {
    "results2.xlsx 已生成": (OUTPUT_DIR / "results2.xlsx").exists(),
    "时间点数为 10801": len(base_sol.t) == 10801,
    "半径点数为 21": len(base_sol.r) == 21,
    "时间步为 1 s": bool(np.allclose(np.diff(base_sol.t), 1.0)),
    "半径范围为 0~2 cm": bool(np.isclose(base_sol.r[0], 0.0) and np.isclose(base_sol.r[-1], 0.02)),
    "温度结果有限": bool(np.all(np.isfinite(base_sol.T))),
    "含水率结果有限": bool(np.all(np.isfinite(base_sol.C))),
    "含水率全为正": bool(np.all(base_sol.C > 0.0)),
}
check_df = pd.DataFrame({"检查项": list(checks.keys()), "是否通过": list(checks.values())})
save_csv(check_df, "最终输出检查.csv", index=False)

all_outputs = sorted([
    p for p in OUTPUT_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in {".csv", ".xlsx", ".png", ".zip", ".txt"}
])
manifest = pd.DataFrame({
    "相对路径": [str(p.relative_to(OUTPUT_DIR)) for p in all_outputs],
    "文件类型": [p.suffix.lower() for p in all_outputs],
    "文件大小（kB）": [p.stat().st_size / 1024.0 for p in all_outputs],
})
save_csv(manifest, "输出文件清单.csv", index=False)

In [ ]:

# ===== 结果压缩 =====
# 方式一：在 notebook 内直接压缩
zip_base = OUTPUT_DIR / "Q2_competition_final_outputs"
zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=OUTPUT_DIR)
print("已生成压缩包：", zip_path)

# 方式二：提供终端压缩命令
zip_command = "zip -r Q2_competition_final_outputs.zip outputs"
print("终端压缩命令：")
print(zip_command)

cmd_file = OUTPUT_DIR / "压缩命令.txt"
cmd_file.write_text(zip_command + "\n", encoding="utf-8")
print("已保存命令文件：", cmd_file)


# 附：正文图片推荐使用顺序

建议正文按如下顺序组织图片：

1. 图 3：环境边界变化；
2. 图 4：温度场时空演化热力图；
3. 图 5：含水率场时空演化热力图；
4. 图 6：径向温度分布；
5. 图 7：径向含水率分布；
6. 图 8：空间非均匀性演化；
7. 图 9：状态依赖扩散系数时空分布；
8. 图 10–13：参数敏感性分析；
9. 图 14–15：网格收敛性分析；
10. 图 16–19：FVM 与 FDM 误差分析图（正文可选，附录推荐）。

## 最后提示

如果你们下一步还希望继续提升这份 Notebook，我建议优先做：

- 给每张主图增加“图后分析”模板；
- 进一步把图片编号与论文段落一一对应；
- 将部分辅助验证图移入附录版本。
